# Lab 18 — Naive Bayes, Ensembles, and Choosing the Right Model

In this lab you will fit a Naive Bayes classifier as a fast baseline, compare bagging against boosting, decompose a multiclass problem with One-vs-Rest and One-vs-One, and then run an honest model-selection-plus-explainability workflow: cross-validate several candidates, pick a winner without touching the test set early, and use permutation importance to see what the winning model actually relies on.

**Concepts covered:** Bayes' theorem as prior × likelihood → normalized posterior, Laplace smoothing, bagging vs. boosting (variance vs. bias reduction), One-vs-Rest vs. One-vs-One multiclass decomposition, honest cross-validated model selection, and model-agnostic permutation importance.

**Reference working sessions:**
- `working-sessions/supervised/12_naive_bayes.ipynb`
- `working-sessions/supervised/13_ensemble_methods.ipynb`
- `working-sessions/supervised/14_multiclass_classification.ipynb`
- `working-sessions/model_evaluation/06_model_selection.ipynb`
- `working-sessions/model_evaluation/09_model_explainability.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.datasets import make_classification
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.multiclass import OneVsRestClassifier, OneVsOneClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance

from tkh_utils import (
    PALETTE, FONT, base_layout,
    check_answer, make_answer_key, make_grading_summary,
    load_heart_disease,
)

_ak = make_answer_key({
    'q1': 'B',
    'q2': 'A',
    'q3': 'C',
    'q4': 'D',
})

---
## Section A — Multiple Choice

Fill in each answer variable with the letter of the best answer (A, B, C, or D).

In [ ]:
# Q1 — In the Bayes calculator widget, Naive Bayes computes an
# "unnormalized score" for each class by multiplying the prior by every
# feature's likelihood. What step converts these two scores into an
# actual probability that sums to 1?
#
#   A) Taking the log of each score
#   B) Dividing each score by the sum of both scores
#   C) Multiplying both scores by the number of features
#   D) Subtracting the smaller score from the larger one

q1_answer = "___"  # Replace with A, B, C, or D

assert q1_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q1_answer, _ak['q1']), \
    "Not quite — revisit working-sessions/supervised/12_naive_bayes.ipynb " \
    "and the Bayes calculator widget's \"What's happening\" section."
print("✓ Question 1 correct!")

In [ ]:
# Q2 — In the Laplace smoothing widget, what happens to every category's
# estimated probability as you raise the smoothing strength alpha to a
# very large value?
#
#   A) Every probability converges toward a uniform distribution across
#      that feature's categories, regardless of what the training data
#      shows
#   B) Every probability goes to exactly zero
#   C) Every probability converges toward the class prior
#   D) The model throws a divide-by-zero error

q2_answer = "___"  # Replace with A, B, C, or D

assert q2_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q2_answer, _ak['q2']), \
    "Not quite — revisit working-sessions/supervised/12_naive_bayes.ipynb " \
    "and the Laplace smoothing widget."
print("✓ Question 2 correct!")

In [ ]:
# Q3 — Bagging and Boosting both combine many decision trees, but they do
# it fundamentally differently. Which statement correctly distinguishes
# them?
#
#   A) Bagging trains trees sequentially on the residuals of the previous
#      tree; Boosting trains trees independently on bootstrap samples
#   B) Bagging and Boosting are the same algorithm under two different
#      names
#   C) Bagging trains trees independently on bootstrap-resampled data and
#      reduces variance; Boosting trains trees sequentially, each
#      correcting the previous ensemble's errors, and can reduce both
#      bias and variance
#   D) Bagging always outperforms Boosting because it can be parallelized

q3_answer = "___"  # Replace with A, B, C, or D

assert q3_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q3_answer, _ak['q3']), \
    "Not quite — revisit working-sessions/supervised/13_ensemble_methods.ipynb " \
    "and the \"What's happening?\" section."
print("✓ Question 3 correct!")

In [ ]:
# Q4 — For a classification problem with C=5 classes, how many binary
# classifiers does One-vs-One (OvO) train, and how does that compare to
# One-vs-Rest (OvR)?
#
#   A) OvO trains 5 classifiers, the same as OvR
#   B) OvO trains only 1 joint classifier, while OvR trains 5
#   C) OvO trains 25 classifiers, one for every ordered pair of classes
#   D) OvO trains C(C-1)/2 = 10 classifiers, more than OvR's 5, but each
#      is trained on a more balanced two-class subset

q4_answer = "___"  # Replace with A, B, C, or D

assert q4_answer != "___", "Don't forget to fill in your answer!"
assert check_answer(q4_answer, _ak['q4']), \
    "Not quite — revisit working-sessions/supervised/14_multiclass_classification.ipynb " \
    "and the \"What's happening?\" section."
print("✓ Question 4 correct!")

In [ ]:
make_grading_summary([
    (q1_answer, _ak['q1'], "Q1: How Naive Bayes normalizes scores into a posterior"),
    (q2_answer, _ak['q2'], "Q2: What high smoothing does to category probabilities"),
    (q3_answer, _ak['q3'], "Q3: The fundamental difference between bagging and boosting"),
    (q4_answer, _ak['q4'], "Q4: How many classifiers OvO trains compared to OvR"),
], total=4)

---
## Section B — Coding Exercises

The three exercises below fit Naive Bayes as a fast baseline, compare a bagging ensemble against a boosting ensemble, and decompose a synthetic 3-class problem with One-vs-Rest and One-vs-One.

In [ ]:
# Shared setup — run this before the exercises
X, y = load_heart_disease()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Dataset shape:", X.shape)
print("Training samples:", X_train.shape[0], " Test samples:", X_test.shape[0])

### B1 — Naive Bayes as a fast baseline

Fit a Gaussian Naive Bayes classifier and a single Decision Tree on the same data, and compare their test-set accuracy.

In [ ]:
# B1 — Fit Naive Bayes and a single Decision Tree, compare test accuracy
nb_model = ___()   # YOUR CODE — the Gaussian Naive Bayes classifier
nb_model.fit(___, ___)   # YOUR CODE — the training features and training target
nb_test_acc = accuracy_score(___, nb_model.predict(X_test))   # YOUR CODE — the true test labels

tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)
tree_test_acc = accuracy_score(y_test, tree_model.predict(X_test))

print(f"Naive Bayes:   test accuracy = {nb_test_acc:.3f}")
print(f"Decision Tree: test accuracy = {tree_test_acc:.3f}")

# --- checks ---
assert 0.6 < nb_test_acc < 1.0, \
    "Naive Bayes test accuracy should be a plausible score, well above chance (0.5)"
assert isinstance(nb_model, GaussianNB)
print("✓ B1 complete!")

### B2 — Bagging vs. Boosting

Fit a Random Forest (bagging) and a Gradient Boosting model (boosting) with matched `n_estimators`, and compare their test-set accuracy against the single tree from B1.

In [ ]:
# B2 — Random Forest (bagging) vs. Gradient Boosting (boosting)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(___, ___)   # YOUR CODE — the training features and training target

gb_model = ___(n_estimators=100, random_state=42)   # YOUR CODE — the gradient boosting classifier for this exercise
gb_model.fit(X_train, y_train)

rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test))
gb_test_acc = accuracy_score(___, gb_model.predict(___))   # YOUR CODE — the true test labels; the test features

print(f"Random Forest (bagging):     test accuracy = {rf_test_acc:.3f}")
print(f"Gradient Boosting (boosting): test accuracy = {gb_test_acc:.3f}")
print(f"Single tree baseline (B1):    test accuracy = {tree_test_acc:.3f}")

# --- checks ---
assert rf_test_acc >= tree_test_acc - 0.05, \
    "Random Forest should be roughly as good as, or better than, a single tree"
assert gb_test_acc >= tree_test_acc - 0.05, \
    "Gradient Boosting should be roughly as good as, or better than, a single tree"
print("✓ B2 complete!")

### B3 — One-vs-Rest vs. One-vs-One

On a synthetic 3-class problem, fit a One-vs-Rest wrapper and a One-vs-One wrapper around the same base classifier, and compare how many classifiers each trains.

In [ ]:
# B3 — OvR vs. OvO on a synthetic 3-class problem
X_mc, y_mc = make_classification(
    n_samples=300, n_features=6, n_informative=4, n_classes=3, random_state=42
)
X_mc_train, X_mc_test, y_mc_train, y_mc_test = train_test_split(
    X_mc, y_mc, test_size=0.2, random_state=42, stratify=y_mc
)

ovr_model = ___(LogisticRegression(max_iter=500, random_state=42))   # YOUR CODE — the multiclass wrapper that trains one classifier per class vs. the rest
ovr_model.fit(___, ___)   # YOUR CODE — the training features and training target

ovo_model = OneVsOneClassifier(LogisticRegression(max_iter=500, random_state=42))
ovo_model.fit(X_mc_train, y_mc_train)

ovr_test_acc = accuracy_score(y_mc_test, ovr_model.predict(X_mc_test))
ovo_test_acc = accuracy_score(___, ovo_model.predict(___))   # YOUR CODE — the true test labels; the test features

print(f"One-vs-Rest: {len(ovr_model.estimators_)} classifiers trained, test accuracy = {ovr_test_acc:.3f}")
print(f"One-vs-One:  {len(ovo_model.estimators_)} classifiers trained, test accuracy = {ovo_test_acc:.3f}")

# --- checks ---
assert len(ovr_model.estimators_) == 3
assert len(ovo_model.estimators_) == 3
assert ovr_test_acc > 0.5 and ovo_test_acc > 0.5, \
    "Both strategies should score well above chance (0.33 for 3 classes)"
print("✓ B3 complete!")

---
## Section C — Applied Problem

A hospital wants an honest answer to two questions: which of several candidate algorithms should be used to predict heart disease, and once you've picked one, which features is it actually relying on? Compare Naive Bayes, Random Forest, and Gradient Boosting via cross-validation on the training set only, pick the best by mean cross-validated accuracy, evaluate it exactly once on the held-out test set, and then use permutation importance — which works the same way regardless of which model won — to see what the winning model is actually using.

In [ ]:
# Section C — Cross-validated model selection, then explain the winner

# --- Step 1: Load and split ---
X_c, y_c = load_heart_disease()
X_c_train, X_c_test, y_c_train, y_c_test = train_test_split(
    ___, ___, test_size=___, random_state=___, stratify=___
    # YOUR CODE — split the features and target, holding out 20% of the
    # data with a fixed seed for reproducibility, preserving the class
    # balance
)

# --- Step 2: Compare candidates via cross-validation on the training set only ---
candidates = {
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cv_results = {}
for name, model in candidates.items():
    scores = cross_val_score(___, X_c_train, y_c_train, cv=5, scoring='accuracy')
    # YOUR CODE — the candidate model for this iteration of the loop
    cv_results[name] = scores.mean()
    print(f"{name}: mean CV accuracy = {scores.mean():.3f}")

# --- Step 3: Pick the winner, fit on the full training set, evaluate once ---
best_name = max(cv_results, key=cv_results.get)
best_model = candidates[best_name]
best_model.___(X_c_train, y_c_train)   # YOUR CODE — the method that trains a model on data

final_test_acc = accuracy_score(y_c_test, best_model.predict(___))   # YOUR CODE — the held-out test features
print(f"\nWinner: {best_name} (test accuracy = {final_test_acc:.3f})")

# --- Step 4: Explain the winner with permutation importance ---
perm = permutation_importance(
    ___, X_c_test, y_c_test, n_repeats=10, random_state=42, scoring='accuracy'
)   # YOUR CODE — the fitted winning model from Step 3

importance_df = pd.DataFrame({
    "feature": X_c.columns,
    "importance": perm.importances_mean,
}).sort_values("importance", ascending=False)

print("\nTop 5 most important features:")
print(importance_df.head(5).round(4))

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=importance_df.head(10), x="importance", y="feature",
            color=PALETTE["primary"], ax=ax)
ax.set_title(f"Permutation Importance — {best_name}")
ax.set_xlabel("Mean accuracy drop when shuffled")
plt.tight_layout()
plt.show()

# --- checks ---
assert final_test_acc > 0.7, \
    "The winning model's held-out test accuracy should be a plausible score, well above chance"
assert len(importance_df) == X_c.shape[1]
print("✓ Section C complete!")
print(f"  Best candidate by cross-validation: {best_name}")
print(f"  Most important feature: {importance_df.iloc[0]['feature']}")

---

## Section D — Reflection

These questions are for reflection. Edit the markdown cells below each question to write your response. There are no wrong answers, we are looking for thoughtful engagement with what you have learned. Your instructor may review these.

**Question D1**

You're building a spam filter that needs to retrain every hour on a stream of new labeled emails, on a machine with very limited memory. Would you reach for Naive Bayes or a Random Forest? Justify your answer using at least one specific property of Naive Bayes covered in `working-sessions/supervised/12_naive_bayes.ipynb`.

*Your response here...*

**Question D2**

In Section C, you cross-validated three candidate models and picked the winner by mean CV accuracy alone. Suppose the hospital instead told you that missing an actual disease case (a false negative) is far more costly than a false alarm. Would cross-validated accuracy still be the right metric to pick the winner by? What would you use instead, and why? Reference `working-sessions/model_evaluation/06_model_selection.ipynb` and the classification metrics you covered earlier in the course.

*Your response here...*

**Question D3**

Permutation importance in Section C works the same way — shuffle one feature, measure the score drop — regardless of which model won the comparison. Why does that model-agnostic property matter for a hospital that might swap the underlying model (say, from Random Forest to Gradient Boosting) as new data comes in? Reference `working-sessions/model_evaluation/09_model_explainability.ipynb` in your answer.

*Your response here...*